# Gradients — the method registry, end to end

Every engine differentiates the primitives it has built through one call,
`Engine.run_gradient(params, method=..., options=...)`, and every variational
algorithm takes the same method names through its `gradient=` argument.

This notebook is the executable companion to the `Gradients` chapter of the
docs: which methods exist, that they agree with calculus, what each costs,
what they return, and what they refuse.

In [ ]:
import numpy as np

import qarpx as qx
from qarp import EXACT
from qarp.algorithms import PauliAveraging, StateVector, VQE
from qarp.blocks import SimpleBlock
from qarp.engines import QarpEngine
from qarp.errors import CapabilityError
from qarp.optimizers import ScipyOptimizer

a, b = qx.Param.symbol("a"), qx.Param.symbol("b")

ansatz = SimpleBlock(2, name="ansatz")
ansatz.ry(0, a).ry(1, b).cx(0, 1)
ansatz.build()

# .symbols is canonically sorted; never hand-zip against another list.
print("symbols:", ansatz.symbols)

## An independent oracle

A gradient that only agrees with another qarp gradient proves nothing. The
observable below has a closed form we can derive by hand, by pushing each
Pauli back through the `CX` (Heisenberg picture) until it acts on a product
state:

| term | `CX†(·)CX` | value on `Ry(a)|0⟩ ⊗ Ry(b)|0⟩` |
|---|---|---|
| `Z0`   | `Z0`    | `cos a` |
| `Z1`   | `Z0 Z1` | `cos a · cos b` |
| `Z0Z1` | `Z1`    | `cos b` |

so for `H = Z0 + Z1 + ½ Z0Z1`

$$E(a,b) = \cos a + \cos a \cos b + \tfrac{1}{2}\cos b$$

and the gradient is elementary calculus — no simulator involved.

In [ ]:
hamiltonian = qx.QubitOperator("Z0") + qx.QubitOperator("Z1") + qx.QubitOperator("Z0 Z1", 0.5)


def exact_energy(av, bv):
    return np.cos(av) + np.cos(av) * np.cos(bv) + 0.5 * np.cos(bv)


def exact_gradient(av, bv):
    # d/da, d/db of exact_energy; columns in ansatz.symbols order ("a", "b").
    return np.array(
        [
            -np.sin(av) * (1.0 + np.cos(bv)),
            -np.cos(av) * np.sin(bv) - 0.5 * np.sin(bv),
        ]
    )


engine = QarpEngine()
engine.build([StateVector(ket=ansatz, operator=hamiltonian)])

point = ansatz.parameter_map([0.4, -0.7])
print("energy    :", engine.run(point)[0].real)
print("analytic  :", exact_energy(0.4, -0.7))

## The four methods against that oracle

`"default"` is the engine's *policy*, not a method: `QarpEngine` resolves it
to `"adjoint"` for an eligible `StateVector` and to `"parameter-shift"`
otherwise. The two analytic methods are exact; the two estimators are not,
so they get the tolerance their own truncation error earns.

In [ ]:
reference = exact_gradient(0.4, -0.7)

runs = [
    ("default", None, 1e-10),
    ("adjoint", None, 1e-10),
    ("parameter-shift", None, 1e-10),
    ("finite-diff", {"fd_eps": 1e-6}, 1e-8),
    ("spsa", {"num_spsa": 400, "spsa_seed": 7}, 2e-1),
]

for method, options, atol in runs:
    grad = engine.run_gradient(point, method=method, options=options)[0]
    err = np.max(np.abs(grad - reference))
    print(f"{method:16s} {np.array2string(grad, precision=6):32s} max|err| = {err:.2e}")
    assert err < atol, (method, err)

print("\nanalytic       ", np.array2string(reference, precision=6))

## Options

`"finite-diff"` takes `fd_eps` (step, default `1e-5`) and `fd_order`
(`1` forward, `2` central — the default). `"spsa"` takes `num_spsa`,
`spsa_c0` and `spsa_seed`.

Central differences are `O(h²)` and forward are `O(h)`, which is visible as
soon as you compare them at the same step.

In [ ]:
for order in (1, 2):
    errs = []
    for eps in (1e-2, 1e-3):
        grad = engine.run_gradient(
            point, method="finite-diff", options={"fd_eps": eps, "fd_order": order}
        )[0]
        errs.append(np.max(np.abs(grad - reference)))
    ratio = errs[0] / errs[1]
    print(f"fd_order={order}  err(1e-2)={errs[0]:.2e}  err(1e-3)={errs[1]:.2e}  ratio={ratio:7.1f}")

# Forward differences lose one order per step decade, central differences two.
print("\nexpected ratios: fd_order=1 -> ~10, fd_order=2 -> ~100")

In [ ]:
# spsa_seed pins the Rademacher draw: same seed, same estimate, bit for bit.
opts = {"num_spsa": 32, "spsa_c0": 0.05, "spsa_seed": 1234}
first = engine.run_gradient(point, method="spsa", options=opts)[0]
again = engine.run_gradient(point, method="spsa", options=opts)[0]
other = engine.run_gradient(point, method="spsa", options={**opts, "spsa_seed": 4321})[0]

print("same seed reproduces :", np.array_equal(first, again))
print("different seed differs:", not np.array_equal(first, other))
assert np.array_equal(first, again)

## The return contract

`run_gradient` returns **one array per built primitive, in build order**. Each
array has one column per entry of `params`, in that mapping's **insertion
order** — for every method. Build the mapping with `block.parameter_map()` (or
from `block.symbols`) and the canonical order comes back.

Never hand-zip a parameter vector against any other list.

In [ ]:
forward = ansatz.parameter_map([0.4, -0.7])                 # ("a", "b")
reversed_map = {"b": -0.7, "a": 0.4}                        # insertion order ("b", "a")

g_fwd = engine.run_gradient(forward)[0]
g_rev = engine.run_gradient(reversed_map)[0]

print("keys", tuple(forward), "->", np.array2string(g_fwd, precision=6))
print("keys", tuple(reversed_map), "->", np.array2string(g_rev, precision=6))
assert np.allclose(g_fwd, g_rev[::-1], atol=1e-10)

The dtype is decided by the primitive's **target**, never by the numeric type
of one evaluation: `float64` for an expectation value over a `QubitOperator`
and for an overlap, `complex128` for a transition amplitude.

There is one deliberate asymmetry worth knowing. A `StateVector` *overlap*
differentiates `|⟨bra|ket⟩|²` even though `run()` returns the complex
amplitude `⟨bra|ket⟩` — that squared quantity is what the VQD family's
deflation penalties need.

In [ ]:
fixed = SimpleBlock(2, name="fixed")
fixed.h(0).cx(0, 1)
fixed.build()

overlap = StateVector(bra=fixed, ket=ansatz)
transition = StateVector(bra=fixed, ket=ansatz, operator=hamiltonian)

probe = QarpEngine()
probe.build([overlap, transition])
grads = probe.run_gradient(point)

for prim, g in zip((overlap, transition), grads, strict=True):
    print(f"{prim.target.name:22s} gradient_kind={prim.gradient_kind:16s} dtype={g.dtype}")

In [ ]:
# The overlap's gradient is d|<bra|ket>|^2/dtheta, checked against a central
# difference of the *squared* modulus -- the objective, not the return value.
eps = 1e-6
squared = []
for sign in (+1, -1):
    shifted = ansatz.parameter_map([0.4 + sign * eps, -0.7])
    squared.append(abs(probe.run(shifted)[0]) ** 2)

fd_da = (squared[0] - squared[1]) / (2 * eps)
print("run() returns the amplitude :", probe.run(point)[0])
print("d|<bra|ket>|^2/da  reported :", grads[0][0])
print("                        fd  :", fd_da)
assert np.isclose(grads[0][0], fd_da, atol=1e-7)

## Repeated symbols

A symbol that appears in several gates — a UCC or Trotter ansatz, a compound
angle from the optimizer's rotation merge — is shifted **one occurrence at a
time** and the contributions summed. Sharing is exact, not approximated.

With `a` driving both rotations the oracle collapses to
`E(a) = cos a + cos²a + ½ cos a`, so `dE/da = −(3/2)·sin a − 2·cos a·sin a`.

In [ ]:
shared = SimpleBlock(2, name="shared")
shared.ry(0, a).ry(1, a).cx(0, 1)
shared.build()

shared_engine = QarpEngine()
shared_engine.build([StateVector(ket=shared, operator=hamiltonian)])

theta = 0.37
shared_point = shared.parameter_map([theta])
expected = -1.5 * np.sin(theta) - 2.0 * np.cos(theta) * np.sin(theta)

for method in ("adjoint", "parameter-shift", "finite-diff"):
    value = shared_engine.run_gradient(shared_point, method=method)[0][0]
    print(f"{method:16s} {value: .10f}   analytic {expected: .10f}")
    assert np.isclose(value, expected, atol=1e-6)

## What is refused, and how

Two names are reserved for later releases so that the spelling is stable now:
`"hadamard"` and `"metric-tensor"`. They raise `CapabilityError`. An unknown
string is a `ValueError` — a typo fails at the call, never as a silent zero.

In [ ]:
for method in ("hadamard", "metric-tensor"):
    try:
        engine.run_gradient(point, method=method)
    except CapabilityError as exc:
        print(f"{method:16s} CapabilityError: {exc}")

try:
    engine.run_gradient(point, method="paramter-shift")   # typo
except ValueError as exc:
    print(f"{'paramter-shift':16s} ValueError: {exc}")

A method an engine does not declare is also a `CapabilityError`, and it names
an engine that has the method. `QarpEngine.gradient_methods` is the whole set;
`CudaqEngine` declares the same set minus `"adjoint"` (a GPU adjoint is
post-release work), so `"adjoint"` there points you back here.

Noise removes the adjoint too — amplitudes are undefined under a noise model,
so `"default"` falls through to the sampled shift.

In [ ]:
print("QarpEngine.gradient_methods:", sorted(QarpEngine.gradient_methods))

Finally, not every primitive can be differentiated analytically. Each declares
a `gradient_kind` saying what the shift rules may assume about `run()` as a
function of each circuit's state: `"expectation"`, `"amplitude"`,
`"squared_overlap"`, or `"none"`.

A `"none"` primitive — `Sampler`, the classical-shadow protocols whose
median-of-means estimator is not linear, and the projected VQE objective
which is a *ratio* — refuses `"parameter-shift"` and keeps `"finite-diff"`.
See `mwe_projected_vqe.ipynb` for that case in a working algorithm.

## Through a variational algorithm

`VQE`, `VQD`, `SSVQE`, `QAOA`, `PCE` and the ADAPT family take
`gradient=True` (the engine's `"default"`), `gradient=False` (a
gradient-free optimizer), or a method name.

A sampled primitive with a gradient is no longer refused at construction — it
differentiates through the batched shift, one shifted point per parameter set
of the sweep.

In [ ]:
vqe = VQE(
    operator=hamiltonian,
    ket=ansatz,
    gradient="parameter-shift",
    primitive=PauliAveraging(n_shots=EXACT),
    initial_parameters=[0.4, -0.7],
    optimizer=ScipyOptimizer("L-BFGS-B", options={"maxiter": 200}),
)
vqe.build()
vqe.suppress_success_message = True

energy, params = vqe.run()

# Oracle: with u = cos a and v = cos b, E = u(1 + v) + v/2. Since 1 + v >= 0 the
# minimum takes u = -1, leaving -1 - v/2, which is smallest at v = 1. So the
# minimum is E = -3/2 at (a, b) = (pi, 0).
print("VQE energy  :", energy)
print("analytic min:", exact_energy(np.pi, 0.0))
assert np.isclose(energy, -1.5, atol=1e-6)